# Data Sources - Classification and regression - Clustering

## Data Sources

### LIBSVM: A Library for Support Vector Machines
https://www.csie.ntu.edu.tw/~cjlin/libsvm/

- LIBSVM is a popular open source library used to train models for classification, regression.

- Written in C++, has C API and ports in Java, Matlab, Python 

See for a list https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/ 

##### Spark LIBSVM data source
https://spark.apache.org/docs/latest/ml-datasource.html#libsvm-data-source

This LIBSVM data source is used to load ‘libsvm’ type files from a directory. The loaded DataFrame has two columns: label containing labels stored as doubles and features containing feature vectors stored as Vectors. The schemas of the columns are:

- label: DoubleType (represents the instance label)
- features: VectorUDT (represents the feature vector)

##### Spark Sample Libsvm data

Load Training Data

In [0]:
# Load training data
training = spark.read.format("libsvm").load("/databricks-datasets/definitive-guide/data/sample_libsvm_data.txt")
training.show(1, truncate=False)

+-----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Showing a single row we have a Sparse Vector representation using col:val

In [0]:
training.head(1)

[Row(label=0.0, features=SparseVector(692, {127: 51.0, 128: 159.0, 129: 253.0, 130: 159.0, 131: 50.0, 154: 48.0, 155: 238.0, 156: 252.0, 157: 252.0, 158: 252.0, 159: 237.0, 181: 54.0, 182: 227.0, 183: 253.0, 184: 252.0, 185: 239.0, 186: 233.0, 187: 252.0, 188: 57.0, 189: 6.0, 207: 10.0, 208: 60.0, 209: 224.0, 210: 252.0, 211: 253.0, 212: 252.0, 213: 202.0, 214: 84.0, 215: 252.0, 216: 253.0, 217: 122.0, 235: 163.0, 236: 252.0, 237: 252.0, 238: 252.0, 239: 253.0, 240: 252.0, 241: 252.0, 242: 96.0, 243: 189.0, 244: 253.0, 245: 167.0, 262: 51.0, 263: 238.0, 264: 253.0, 265: 253.0, 266: 190.0, 267: 114.0, 268: 253.0, 269: 228.0, 270: 47.0, 271: 79.0, 272: 255.0, 273: 168.0, 289: 48.0, 290: 238.0, 291: 252.0, 292: 252.0, 293: 179.0, 294: 12.0, 295: 75.0, 296: 121.0, 297: 21.0, 300: 253.0, 301: 243.0, 302: 50.0, 316: 38.0, 317: 165.0, 318: 253.0, 319: 233.0, 320: 208.0, 321: 84.0, 328: 253.0, 329: 252.0, 330: 165.0, 343: 7.0, 344: 178.0, 345: 252.0, 346: 240.0, 347: 71.0, 348: 19.0, 349: 28.0

If we want to get the dense format

In [0]:
from pyspark.ml.functions import vector_to_array

training.select(vector_to_array('features')).show(1,truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Let's get the size of dataset

In [0]:
training.count()

100

##### Challenge
What data are  https://github.com/apache/spark/blob/master/data/mllib/sample_libsvm_data.txt ?

### Image data source
https://spark.apache.org/docs/latest/ml-datasource.html#image-data-source

This image data source is used to load image files from a directory, it can load compressed image (jpeg, png, etc.) into raw image representation via ImageIO in Java library. 

The loaded DataFrame has one StructType column: “image”, containing image data stored as image schema. 

The schema of the image column is:

- origin: StringType (represents the file path of the image)
- height: IntegerType (height of the image)
- width: IntegerType (width of the image)
- nChannels: IntegerType (number of image channels)
- mode: IntegerType (OpenCV-compatible type)
- data: BinaryType (Image bytes in OpenCV-compatible order: row-wise BGR in most cases)

## Classification

### Logistic regression
https://spark.apache.org/docs/latest/ml-classification-regression.html#logistic-regression

Logistic regression is a popular method to predict a categorical response. 

It is a special case of [Generalized Linear models](https://en.wikipedia.org/wiki/Generalized_linear_model) that predicts the probability of the outcomes. 

In spark.ml logistic regression can be used to predict a binary outcome by using binomial logistic regression, or it can be used to predict a multiclass outcome by using multinomial logistic regression. Use the family parameter to select between these two algorithms, or leave it unset and Spark will infer the correct variant.

> Multinomial logistic regression can be used for binary classification by setting the family param to “multinomial”. It will produce two sets of coefficients and two intercepts.

> When fitting LogisticRegressionModel without intercept on dataset with constant nonzero column, Spark MLlib outputs zero coefficients for constant nonzero columns. This behavior is the same as R glmnet but different from LIBSVM.

#### Binomial logistic regression

> Binary logistic regression models the relationship between a set of predictors and a binary response variable. A binary response has only two possible values, such as win and lose. Use a binary regression model to understand how changes in the predictor values are associated with changes in the probability of an event occurring.

https://statisticsbyjim.com/glossary/binary-logistic-regression/

### Logistic Regression in Pyspark

https://spark.apache.org/docs/latest/ml-classification-regression.html#logistic-regression

https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html

#### Example with spark sample dataset

Importing Library

In [0]:
from pyspark.ml.classification import LogisticRegression

Create the Logistic Regression Object (act as an estimator)

In [0]:
lr = LogisticRegression(maxIter=10, regParam=0.3, elasticNetParam=0.8)

In [0]:
lr

LogisticRegression_e0e11a8c561c

Run the fit method passing the training set as value

In [0]:
# Fit the model
lrModel = lr.fit(training)

The output is a model that can be used in trasformation

In [0]:
lrModel

LogisticRegressionModel: uid=LogisticRegression_4b3235d43780, numClasses=2, numFeatures=692

[LogisticRegressionModel](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegressionModel.html) implements standard methods to transform, analyse and save the model

In [0]:
# Print the coefficients and intercept for logistic regression
print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

Coefficients: (692,[272,300,323,350,351,378,379,405,406,407,428,433,434,435,455,456,461,462,483,484,489,490,496,511,512,517,539,540,568],[-7.520689871384157e-05,-8.11577314684704e-05,3.814692771846389e-05,0.0003776490540424341,0.0003405148366194407,0.0005514455157343111,0.00040853861160969167,0.00041974673327494573,0.0008119171358670032,0.0005027708372668752,-2.392926040660149e-05,0.0005745048020902299,0.000903754642680371,7.818229700243959e-05,-2.17875519529124e-05,-3.402165821789581e-05,0.0004966517360637634,0.0008190557828370371,-8.017982139522661e-05,-2.743169403783574e-05,0.00048108322262389896,0.00048408017626778744,-8.926472920010679e-06,-0.0003414881233042728,-8.950592574121448e-05,0.0004864546911689218,-8.478698005186158e-05,-0.0004234783215831764,-7.296535777631296e-05])
Intercept: -0.5991460286401442


Many model have the the summary attribute containing information about the train

In [0]:
# Extract the summary from the returned LogisticRegressionModel instance trained
# in the earlier example
trainingSummary = lrModel.summary
trainingSummary

Out[6]: <pyspark.ml.classification.BinaryLogisticRegressionTrainingSummary at 0x7fad2cb62040>

The lrSummary is a java object containing the training result of the models
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.BinaryLogisticRegressionTrainingSummary.html

In the following some examples of values

Model accuracy

In [0]:
# Get the accuracy of the model
trainingSummary.accuracy

Out[7]: 0.99

Objective History

In [0]:
# Obtain the objective per iteration (defined in maxIter)
objectiveHistory = trainingSummary.objectiveHistory

objectiveHistory

Out[8]: [0.6833149135741672,
 0.6661906127558116,
 0.6207433672479604,
 0.613154125312387,
 0.6059149689952394,
 0.5923656241678249,
 0.5898233082838019,
 0.5868012627420284,
 0.5844432058719141,
 0.5830790068041745,
 0.5807015754032354]

Receiver Operating Characteristic 



In [0]:
# Obtain the receiver-operating characteristic as a dataframe and areaUnderROC.
trainingSummary.roc.show()


+---+--------------------+
|FPR|                 TPR|
+---+--------------------+
|0.0|                 0.0|
|0.0|0.017543859649122806|
|0.0| 0.03508771929824561|
|0.0| 0.05263157894736842|
|0.0| 0.07017543859649122|
|0.0| 0.08771929824561403|
|0.0| 0.10526315789473684|
|0.0| 0.12280701754385964|
|0.0| 0.14035087719298245|
|0.0| 0.15789473684210525|
|0.0| 0.17543859649122806|
|0.0| 0.19298245614035087|
|0.0| 0.21052631578947367|
|0.0| 0.22807017543859648|
|0.0| 0.24561403508771928|
|0.0|  0.2631578947368421|
|0.0|  0.2807017543859649|
|0.0|  0.2982456140350877|
|0.0|  0.3157894736842105|
|0.0|  0.3333333333333333|
+---+--------------------+
only showing top 20 rows

areaUnderROC: 1.0


In [0]:
print("areaUnderROC: " + str(trainingSummary.areaUnderROC))

areaUnderROC: 1.0


In [0]:
# Set the model threshold to maximize F-Measure
fMeasure = trainingSummary.fMeasureByThreshold
maxFMeasure = fMeasure.groupBy().max('F-Measure').select('max(F-Measure)').head()
bestThreshold = fMeasure.where(fMeasure['F-Measure'] == maxFMeasure['max(F-Measure)']) \
    .select('threshold').head()['threshold']
lr.setThreshold(bestThreshold)

Out[11]: LogisticRegression_9cca801a23c2

In [0]:
fMeasure = trainingSummary.fMeasureByThreshold

In [0]:
# Fit the model
lrModelOpt = lr.fit(training)
lrModelOpt.summary.accuracy

Out[12]: 1.0

#### Example from scratch

## Clustering
https://spark.apache.org/docs/latest/ml-clustering.html#clustering

### K-means
k-means is one of the most commonly used clustering algorithms that clusters the data points into a predefined number of clusters. The MLlib implementation includes a parallelized variant of the k-means++ method called kmeans||.

KMeans is implemented as an Estimator and generates a KMeansModel as the base model.

**Input Columns**

|Param name	|  Type(s)	| Default	| Description  | 
|-----------|  ------   | ----------| ----------   |
|featuresCol|  Vector	|"features"	|Feature vector|

**Output Columns**

|Param name	|  Type(s)	| Default	| Description  | 
|-----------|  ------   | ----------| ----------   |
|predictionCol|	Int|	"prediction"|	Predicted cluster center

In [0]:
!curl https://raw.githubusercontent.com/apache/spark/master/data/mllib/sample_kmeans_data.txt --output /tmp/sample_kmeans_data.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   120  100   120    0     0    461      0 --:--:-- --:--:-- --:--:--   461


In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Loads data.
dataset = spark.read.format("libsvm").load("file:/tmp/sample_kmeans_data.txt")

#dataset.show(truncate=False)
dataset.count()

6

In [0]:
# Trains a k-means model.
kmeans = KMeans().setK(2).setSeed(1)
model = kmeans.fit(dataset)

# Make predictions
predictions = model.transform(dataset)
predictions.show()

Uploading artifacts:   0%|          | 0/4 [00:00<?, ?it/s]

+-----+--------------------+----------+
|label|            features|prediction|
+-----+--------------------+----------+
|  0.0|           (3,[],[])|         1|
|  1.0|(3,[0,1,2],[0.1,0...|         1|
|  2.0|(3,[0,1,2],[0.2,0...|         1|
|  3.0|(3,[0,1,2],[9.0,9...|         0|
|  4.0|(3,[0,1,2],[9.1,9...|         0|
|  5.0|(3,[0,1,2],[9.2,9...|         0|
+-----+--------------------+----------+



In [0]:
# Evaluate clustering by computing Silhouette score
evaluator = ClusteringEvaluator()

silhouette = evaluator.evaluate(predictions)
print("Silhouette with squared euclidean distance = " + str(silhouette))

Silhouette with squared euclidean distance = 0.9997530305375207


In [0]:
# Shows the result.
centers = model.clusterCenters()
print("Cluster Centers: ")
for center in centers:
    print(center)

Cluster Centers: 
[9.1 9.1 9.1]
[0.1 0.1 0.1]
